<a href="https://colab.research.google.com/github/MAI-AIIU/advanced-programming/blob/main/assignments/first/src/First_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building an AI-Ready Dataset Using Web Scraping, External Datasets, NumPy, and Pandas.

In this assignment, students will collect data from the web using web scraping and combine it with another external
dataset. They will clean, transform, analyze, and prepare the data using Pandas and NumPy.

## Scraping `books.toscrape.com` website
Loading the required libraries, which I use `requests` to send request and pull data and using `BeautifulSoup` to scrape data from the HTML and CSS data I have recieved via request

#### Load libraries

In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd

#### Define base_url and headers

In [2]:
BASE_URL = "https://books.toscrape.com/"

# we use this header to avoid flagging our request as automated by the website
headers = {
    "User-Agent": "Mozilla/5.0"
}

#### Define mappers to map data from original form to another form
in below example we map ratings from text to numbers

In [3]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

#### Defining global variables

In [4]:
books = []
next_page = BASE_URL
page = 5 # I put this variable to only load first 5 pages.

In [5]:
while next_page:
    print(f"Scraping {next_page}")

    response = requests.get(next_page, headers=headers)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    for book in soup.select("article.product_pod"):

        title = book.h3.a["title"]
        price = book.select_one("p.price_color").text.strip()

        availability = (
            book.select_one("p.instock.availability")
            .get_text(strip=True)
        )

        rating = rating_map[
            book.select_one("p.star-rating")["class"][1]
        ]

        # Product URL
        relative_url = book.h3.a["href"]
        product_url = urljoin(next_page, relative_url)

        # Visit product page to get category
        detail_response = requests.get(product_url, headers=headers)
        detail_response.raise_for_status()

        detail_soup = BeautifulSoup(detail_response.text, "html.parser")

        category = (
            detail_soup.select("ul.breadcrumb li a")[2]
            .get_text(strip=True)
        )

        books.append({
            "Title": title,
            "Price": price,
            "Rating": rating,
            "Availability": availability,
            "Category": category,
            "Product Link": product_url
        })

    # Find next page
    next_button = soup.select_one("li.next a")

    if next_button and page > 1:
        page -= 1
        next_page = urljoin(next_page, next_button["href"])
    else:
        next_page = None

toscrape_books = pd.DataFrame(books)
print(f"Scraped {len(books)} books.")
print("Saved to books.csv.")

Scraping https://books.toscrape.com/
Scraping https://books.toscrape.com/catalogue/page-2.html
Scraping https://books.toscrape.com/catalogue/page-3.html
Scraping https://books.toscrape.com/catalogue/page-4.html
Scraping https://books.toscrape.com/catalogue/page-5.html
Scraped 100 books.
Saved to books.csv.


#### Export to CSV file

In [6]:
toscrape_books.to_csv("toscrape_books.csv", index=False)

## Loading books csv file from Github

Note: As Kaggle required to authenticate before loading dataset, and this dataset was the matched one with the one I scraped from `https://books.toscrape.com/`, so I downloaded its CSV and uploaded to my own Github repo, and put the code below to pull to Colab Notebook.

In [7]:
url = "https://raw.githubusercontent.com/MAI-AIIU/advanced-programming/main/assignments/first/files/github_books.csv"

github_books = pd.read_csv(url)

#### Rename column names to match `toscrape_books` with `github_books`

In [8]:
github_books.columns, toscrape_books.columns # matching both columns

(Index(['index', 'title', 'price', 'rating', 'stock', 'category', 'book_url'], dtype='object'),
 Index(['Title', 'Price', 'Rating', 'Availability', 'Category', 'Product Link'], dtype='object'))

In [9]:
# this will rename and change the order of columns to match
github_books = (
    github_books
    .rename(columns={
        "title": "Title",
        "price": "Price",
        "rating": "Rating",
        "stock": "Availability",
        "category": "Category",
        "book_url": "Product Link"
    })
    [["Title", "Price",  "Rating", "Availability", "Category", "Product Link"]]
)

In [10]:
github_books.columns, toscrape_books.columns # matching both columns

(Index(['Title', 'Price', 'Rating', 'Availability', 'Category', 'Product Link'], dtype='object'),
 Index(['Title', 'Price', 'Rating', 'Availability', 'Category', 'Product Link'], dtype='object'))

#### Export to CSV file

In [11]:
github_books.to_csv("github_books.csv", index=False)

## Pandas Data Cleaning and Preparation

#### Explain both datasets

In [12]:
toscrape_books.head(1), github_books.head(1)

(                  Title    Price  Rating Availability Category  \
 0  A Light in the Attic  Â£51.77       3     In stock   Poetry   
 
                                         Product Link  
 0  https://books.toscrape.com/catalogue/a-light-i...  ,
             Title Price Rating Availability       Category  \
 0  Modern Romance   NaN   Five     In stock  Add a comment   
 
                                         Product Link  
 0  https://books.toscrape.com/catalogue/modern-ro...  )

In [13]:
# prints dataset columns

toscrape_books.columns, github_books.columns

(Index(['Title', 'Price', 'Rating', 'Availability', 'Category', 'Product Link'], dtype='object'),
 Index(['Title', 'Price', 'Rating', 'Availability', 'Category', 'Product Link'], dtype='object'))

In [14]:
# prints datasets information such as columns, non-null counts, data type of each column, memory usage, range of data
toscrape_books.info(), github_books.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Title         100 non-null    object
 1   Price         100 non-null    object
 2   Rating        100 non-null    int64 
 3   Availability  100 non-null    object
 4   Category      100 non-null    object
 5   Product Link  100 non-null    object
dtypes: int64(1), object(5)
memory usage: 4.8+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Title         196 non-null    object
 1   Price         194 non-null    object
 2   Rating        199 non-null    object
 3   Availability  200 non-null    object
 4   Category      199 non-null    object
 5   Product Link  200 non-null    object
dtypes: object(6)
memory usage: 9.5+ KB


(None, None)

In [15]:
# prints dataset shapes, the first number in tuple is number of rows, and the second is number of columns
toscrape_books.shape, github_books.shape

((100, 6), (200, 6))

In [16]:
# print top 3 rows of first dataset and 3 rows from end from second dataset
toscrape_books.head(3), github_books.tail(3)

(                  Title    Price  Rating Availability            Category  \
 0  A Light in the Attic  Â£51.77       3     In stock              Poetry   
 1    Tipping the Velvet  Â£53.74       1     In stock  Historical Fiction   
 2            Soumission  Â£50.10       1     In stock             Fiction   
 
                                         Product Link  
 0  https://books.toscrape.com/catalogue/a-light-i...  
 1  https://books.toscrape.com/catalogue/tipping-t...  
 2  https://books.toscrape.com/catalogue/soumissio...  ,
                                                  Title    Price Rating  \
 197                                        Most Wanted  Â£35.28  Three   
 198                               Love, Lies and Spies  Â£20.55    Two   
 199  How to Speak Golf: An Illustrated Guide to Lin...  Â£58.32   Five   
 
     Availability            Category  \
 197     In stock             Mystery   
 198     In stock  Historical Fiction   
 199     In stock             Defaul

In [17]:
# describe prints some statistics from both datasets like count, mean, std, min, 25%, and also prints count, unique, top, ...
toscrape_books.describe(), github_books.describe()

(           Rating
 count  100.000000
 mean     2.930000
 std      1.423149
 min      1.000000
 25%      2.000000
 50%      3.000000
 75%      4.000000
 max      5.000000,
                  Title    Price Rating Availability Category  \
 count              196      194    199          200      199   
 unique             196      192      5            1       36   
 top     Modern Romance  Â£36.58  Three     In stock  Default   
 freq                 1        2     43          200       36   
 
                                              Product Link  
 count                                                 200  
 unique                                                200  
 top     https://books.toscrape.com/catalogue/modern-ro...  
 freq                                                    1  )

In [20]:
github_books.isna().sum(), toscrape_books.isna().sum()

(Title           4
 Price           6
 Rating          1
 Availability    0
 Category        1
 Product Link    0
 dtype: int64,
 Title           0
 Price           0
 Rating          0
 Availability    0
 Category        0
 Product Link    0
 dtype: int64)

#### Handling missing values

As per above analysis, scrape books don't have any missing value but Github books have some, so  
1: filling missing values in Price and Ratings  
2: dropping records having missing values on Title